# 3. Welcome To Langchain

## 3.0 LLMs and Chat Models
### Models: LLM VS Chat
1. LLM
    - using text-davinci-003
2. Chat
    - using gpt-3.5-turbo.
    - price is 1/10 of text-davinci-003.
    - group of messages
    - reply as a whole conversation.

In [ ]:
from langchain.llms.openai import OpenAI
from langchain.chat_models import ChatOpenAI, ChatAnthropic # ChatAnthropic 같은 다른 모델로 갈아끼울 수 있다.
from langchain.callbacks import StreamingStdOutCallbackHandler # LLM에 발생하는 이벤트(작업 시작, 문자가 생성되기 시작, 에러) 감지

llm = OpenAI()
chat = ChatOpenAI(
    # customize options: 
    temperature=0.1,
    streaming=True, # response 생성 과정을 보여준다.
    callbacks=[StreamingStdOutCallbackHandler()], # response가 생성되면 바로 콘솔에 출력된다.
) 

## 3.1 Predict Messages
### Methods: predict VS predict_messages
- LLM과 Chat 모델 공통적으로 아래 methods를 쓸 수 있다. 비용이 저렴한 관계로 아래 예시부터는 Chat 모델 사용 예정.
1. predict
    - question and get an answer as a text.
2. predict_messages
    - predict chat messages

In [ ]:
# 1. predict
from langchain.chat_models import ChatOpenAI

chat = ChatOpenAI(temperature=0.1)

chat.predict("How many planets are there?")

# 2. predict_messages
from langchain.schema import HumanMessage, AIMessage, SystemMessage # import message constructors.

messages = [
    SystemMessage(content="You are a geography expert. And you only reply in Italian."), # Provide Context Settings to LLM
    AIMessage(content="Ciao, mi chiamo Paolo."),
    HumanMessage(content="What is the distance btw Mexico and Thailand? And also what is your name?"),
] 

chat.predict_messages(messages)

# Customize prompt with placeholder.

messages = [
    SystemMessage(content="You are a geography expert. And you only reply in {language}."),
    AIMessage(content="Ciao, mi chiamo {name}."),
    HumanMessage(content="What is the distance btw {country_a} and {country_b}? And also what is your name?"),
]


## 3.2 Prompt Templates
- 사용하는 이유:
    1) validation
    2) 디스크에 저장하고 불러오기 할 수 있다.
### Prompt Template VS Chat Prompt Template
1. Prompt Template
    - for predicting text
    - template을 string으로 만든다.

2. ChatPromptTemplate
    - template을 message로 만든다.

In [ ]:
# 1. Prompt Template

from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate, ChatPromptTemplate

# 1) make template
template = PromptTemplate.from_template(
    "What is the distance btw {country_a} and {country_b}?"
)

# .from_template이 없다면?
t = PromptTemplate(
    template= "What is the distance btw {country_a} and {country_b}?",
    input_variables=["country"],
)
t.format(country="France")

# 2) format the template and save it as a prompt. => 'What is the distance btw Mexico and Thailand?'
prompt = template.format(country_a = "Mexico", country_b = "Thailand")

# 3) call predict.
chat = ChatOpenAI(temperature=0.1)
chat.predict(prompt)

In [ ]:
# 2. Chat Prompt Template

from langchain.chat_models.openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate

# Using ChatPromptTemplate instead of predict_message's HumanMessage, AIMessage, SystemMessage.
template = ChatPromptTemplate.from_messages([
    ("system", "You are a geography expert. And you only reply in {language}."),
    ("ai", "Ciao, mi chiamo {name}."),
    ("human", "What is the distance btw {country_a} and {country_b}? And also what is your name?")
])

prompt = template.format_messages(
    language = "Greek",
    name = "Socrates",
    country_a = "Mexico",
    country_b = "Thailand",
)

chat = ChatOpenAI(temperature=0.1)
chat.predict_messages(prompt)

AIMessage(content='Γεια σας! Η απόσταση μεταξύ του Μεξικού και της Ταϊλάνδης είναι περίπου 16.000 χιλιόμετρα. Το όνομά μου είναι Σωκράτης. Πώς μπορώ να βοηθήσω;')

## 3.3 OutputParser
    - LLM의 output을 구조별로 parsing.
    - text로만 응답하는 LLM의 response 타입을 변형할 필요가 있을 때 사용.

In [ ]:
# 1. BaseOutputParser를 확장하여 custom OutputParser를 만든다.
from langchain.schema import BaseOutputParser

class CommaOutputParser(BaseOutputParser):
    def parse(self, text):
        items = text.strip().split(",") # strip: text 앞 뒤의 공백을 없앤다. / split: ","를 기준으로 string을 잘라 array로 변환한다.
        return list(map(str.strip, items))

# 2. make chat model - make template - message format - predict
from langchain.prompts import ChatPromptTemplate
from langchain.chat_models.openai import ChatOpenAI

chat = ChatOpenAI()

template = ChatPromptTemplate.from_messages([
    ("system", "You are a list generating machine. Everything you are asked will be answered with a comma separated list of max_items {max_items} in lowercase. Do NOT reply with anything else."),
    ("human", "{question}"),
])

prompt = template.format_messages(
    max_items = 10,
    question = "What are the colours",
)

result = chat.predict_messages(prompt)

# 3. OutputParser 사용
p = CommaOutputParser()
p.parse(result.content)

## 3.3 LCEL(LangChain Expression Language)
    - Save a lot of codes. template, OutputParser, Chat model만 필요로 한다.
    - 다양한 templates + LLM 호출 + 다양한 response를 함께 사용할 수 있게 해준다.
### Chain
    - A group of components next to each other.
    - All components run together.
    - Each component take an input and give an output.
    - output of each component is given to the next component on the line until the chain is finished.
    - Human get the final output.
    - 체인끼리도 결합 가능.
    - 체인이 가질 수 있는 components:
|Component|Input Type|Output Type|
|---------|----------|-----------|
|Prompt|Dictionary|PromptValue(Formatted Prompt)|
|Retriever|Single string|List of Documents|
|LLM|Single string, list of chat messages or a PromptValue|String|
|ChatModel|Single string, list of chat messages or a PromptValue|ChatMessage|
|Agent Tool|Single string, or dictionary, depending on the tool|
|OutputParser|The output of an LLM or ChatModel|

In [ ]:
# What LCEL needed?

# 1. chat model
from langchain.chat_models import ChatOpenAI

chat = ChatOpenAI(temperature=0.1)

# 2. OutputParser
from langchain.schema import BaseOutputParser

class CommaOutputParser(BaseOutputParser):
    def parse(self, text):
        items = text.strip().split(",")
        return list(map(str.strip, items))

# 3. template
from langchain.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages([
    ("system", "You are a list generating machine. Everything you are asked will be answered with a comma separated list of max_items {max_items} in lowercase. Do NOT reply with anything else."),
    ("human", "{question}"),
])

# 4. chain
chain = template | chat | CommaOutputParser()
# internally LangChain calls => template.format_messages | chat.predict | parser.parse

chain.invoke({
    "max_items":5,
    "question":"What are the Pocketmons",
})

In [ ]:
# 3.4 Chaining Chains

from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()],
)

# chain 1
chef_prompt = ChatPromptTemplate.from_messages([
    ("system","You are a world-class baker. You make easy to follow recipes for no-sugar and no-butter breads."),
    ("human","I want to cook {cuisine} food"),
])

chef_chain = chef_prompt | chat

# chain 2
veg_chef_prompt = ChatPromptTemplate.from_messages([
    ("system","You've worked at world-class bakeries around the world. You make the original recipe into the same as renown bakeries' best-selling items."),
    ("human","{recipe}")
])

veg_chain = veg_chef_prompt | chat

# final chain 순서(RunnableMap) == {"recipe": chef_chain.invoke({"cuisine":"indian"})} -> veg_chain.invoke({"recipe": chef_chain result})
final_chain = {"recipe": chef_chain} | veg_chain

final_chain.invoke({
    "cuisine":"indian"
})